# Synthetic fashion SKU demand generator

Produces a panel at grain (sku_id, date)

Two outputs:
  - full dataframe
  - model_ready dataframe

The generated dataset is loaded into a BigQuery for future usage

### libraries

In [15]:
# basic libraries
import numpy as np
import pandas as pd

# big query 
from google.cloud import bigquery
from google.cloud import storage

### Params

A later section adds these below params as a yaml file and stores into the bucket for future uses

In [16]:
SEED = 42
N_SKUS = 300
START_DATE = "2020-01-06"
N_WEEKS = 260 #5 years of dataset

CATEGORIES = ["Tops", "Bottoms", "Dresses", "Outerwear", "Footwear", "Accessories"]
SUB_CATEGORY = {
    "Tops": ["T-Shirt", "Blouse", "Shirt", "Sweater"],
    "Bottoms": ["Jeans", "Trousers", "Shorts", "Skirt"],
    "Dresses": ["Casual Dress", "Evening Dress", "Sundress"],
    "Outerwear": ["Jacket", "Coat", "Parka"],
    "Footwear": ["Sneakers", "Boots", "Sandals", "Heels"],
    "Accessories": ["Bag", "Belt", "Scarf", "Hat"],
}
COLORS = ["Black", "White", "Beige", "Navy", "Red", "Pastel Pink", "Olive", "Grey"]
MATERIALS = ["Cotton", "Denim", "Wool", "Synthetic", "Leather", "Linen"]
BRANDS = ["ZARA", "H&M", "COS", "Mango", "Uniqlo"]
GENDERS = ["Male", "Female", "Unisex"]
SIZE_RANGES = ["XS-S", "S-M", "M-L", "L-XL"]

In [17]:
# category -> (pattern_type, seasonal_peak_week, seasonal_amplitude, weather_sensitivity)
# pattern_type mirrors the TYPE_LABEL_MAP taxonomy from the forecast-framework
# project (trend / seasonal / trend_seasonal / cyclic) so this data can slot
# straight into that pipeline's per-category algorithm pools.
CATEGORY_PROFILE = {
    "Tops":        dict(pattern_type="seasonal",      peak_week=26, amplitude=0.35, weather_sign=+1),
    "Bottoms":     dict(pattern_type="trend",          peak_week=20, amplitude=0.10, weather_sign=0),
    "Dresses":     dict(pattern_type="trend_seasonal", peak_week=24, amplitude=0.50, weather_sign=+1),
    "Outerwear":   dict(pattern_type="seasonal",       peak_week=50, amplitude=0.60, weather_sign=-1),
    "Footwear":    dict(pattern_type="trend",          peak_week=30, amplitude=0.15, weather_sign=0),
    "Accessories": dict(pattern_type="cyclic",         peak_week=None, amplitude=0.20, weather_sign=0),
}
PRICE_ELASTICITY = {  # demand multiplier = (price / base_price) ** -elasticity
    "Tops": 1.2, "Bottoms": 1.0, "Dresses": 1.4,
    "Outerwear": 0.8, "Footwear": 1.1, "Accessories": 0.9,
}

rng = np.random.default_rng(SEED)

### 1. SKU master (static attributes)

In [18]:
def make_sku_master(n_skus: int) -> pd.DataFrame:
    categories = rng.choice(CATEGORIES, size=n_skus)
    rows = []
    for i, cat in enumerate(categories):
        sub_cat = rng.choice(SUB_CATEGORY[cat])
        base_price = rng.uniform(20, 150) if cat != "Outerwear" else rng.uniform(60, 250)

        # 80% of SKUs launch at the very start of history (gives most SKUs a
        # full, stable series); 20% launch later, staggered across the
        # timeline, so lifecycle features (new/growth/mature/decline) have
        # real examples to learn from instead of being constant.
        if rng.random() < 0.8:
            launch_week = 0
        else:
            launch_week = int(rng.integers(1, N_WEEKS - 60))

        rows.append(dict(
            sku_id=f"SKU{i:05d}",
            category=cat,
            sub_category=sub_cat,
            color=rng.choice(COLORS),
            material=rng.choice(MATERIALS),
            brand=rng.choice(BRANDS),
            gender=rng.choice(GENDERS),
            size_range=rng.choice(SIZE_RANGES),
            base_price=round(base_price, 2),
            launch_week=launch_week,
            lead_time_days=int(rng.choice([14, 21, 30, 45])),
            store_count=int(rng.integers(20, 400)),
            channel=rng.choice(["online", "in-store", "both"], p=[0.3, 0.3, 0.4]),
            # base demand level: lognormal so a few SKUs are hits and most
            # are modest -- matches the skewed target distribution EDA
            # step 3 (step3_target_distribution) is written to expect.
            base_demand_level=rng.lognormal(mean=3.0, sigma=0.6),
            quality_score=float(np.clip(rng.normal(4.0, 0.4), 2.5, 5.0)),  # drives avg_rating later
        ))
    return pd.DataFrame(rows)

### 2. Calendar (deterministic from date, shared across all SKUs)

In [19]:
def make_calendar(start_date: str, n_weeks: int) -> pd.DataFrame:
    dates = pd.date_range(start=start_date, periods=n_weeks, freq="W-MON")
    df = pd.DataFrame({"date": dates})
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["is_weekend"] = False  # week-start rows only, N/A at weekly grain

    # simplified fixed holiday weeks (ISO week numbers based on EU holiday season)
    holiday_weeks = {1, 7, 20, 47, 48, 51, 52}
    df["is_holiday"] = df["week_of_year"].isin(holiday_weeks)
    
    # assunimg last 3 weeks of Feb/May/Sep/Nov fashion weeks
    df["fashion_week_flag"] = df["week_of_year"].isin({7, 8, 9, 20, 21, 22, 36, 37, 38, 47, 48, 49})  

    # assuming 3 weeks of August     
    df["back_to_school_flag"] = df["week_of_year"].isin({33, 34, 35})
    
    # first week of month
    df["payday_week_flag"] = df["date"].dt.day <= 7                   
    return df

### 3. Weather (seasonal sinusoid + noise, Northern-hemisphere shaped)

In [20]:
def make_weather(calendar: pd.DataFrame) -> pd.DataFrame:
    df = calendar[["date", "week_of_year"]].copy()
    # temperature peaks ~week 30 (late July), troughs ~week 4 (late Jan)
    seasonal = 15 * np.sin(2 * np.pi * (df["week_of_year"] - 30) / 52)
    df["temperature"] = 12 + seasonal + rng.normal(0, 2.5, len(df))
    df["precipitation"] = np.clip(
        rng.gamma(shape=2.0, scale=8.0, size=len(df)) + 3 * np.sin(2 * np.pi * df["week_of_year"] / 52),
        0, None,
    )
    return df[["date", "temperature", "precipitation"]]

### 4. Price / promo / marketing panel (per SKU, per week)

In [21]:
def make_price_promo_marketing(sku_master: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    panel = sku_master[["sku_id", "base_price"]].merge(calendar[["date"]], how="cross")
    panel = panel.sort_values(["sku_id", "date"]).reset_index(drop=True)
    n = len(panel)

    promo_flag = np.zeros(n, dtype=bool)
    for sku, idx in panel.groupby("sku_id").indices.items():
        idx = np.array(idx)
        t = 0
        while t < len(idx):
            if rng.random() < 0.06:  # ~6% weekly chance a promo run starts
                run_len = int(rng.integers(1, 4))
                promo_flag[idx[t: t + run_len]] = True
                t += run_len
            else:
                t += 1
    panel["promo_flag"] = promo_flag

    panel["discount_pct"] = np.where(
        panel["promo_flag"], rng.uniform(0.1, 0.5, n), 0.0
    )
    panel["markdown_stage"] = pd.cut(
        panel["discount_pct"], bins=[-0.01, 0, 0.15, 0.3, 1.0],
        labels=["none", "light", "moderate", "deep"],
    )
    panel["price"] = (panel["base_price"] * (1 - panel["discount_pct"])).round(2)
    panel["competitor_price_index"] = rng.normal(1.0, 0.05, n).clip(0.8, 1.2)

    # marketing spend correlates with promo (campaigns usually back promos)
    # plus an independent baseline so it's not a pure duplicate of promo_flag.
    panel["marketing_spend"] = (
        rng.gamma(2.0, 150, n) + panel["promo_flag"] * rng.uniform(500, 2000, n)
    ).round(2)
    panel["campaign_flag"] = panel["promo_flag"] & (rng.random(n) < 0.7)

    return panel

### 5. Cross-product: pair each SKU with an in-category substitute

In [22]:
def add_cross_product_features(panel: pd.DataFrame, sku_master: pd.DataFrame) -> pd.DataFrame:
    substitute_map = {}
    for cat, group in sku_master.groupby("category"):
        ids = group["sku_id"].tolist()
        shuffled = rng.permutation(ids)

        for i, sku in enumerate(ids):
            substitute_map[sku] = shuffled[(list(shuffled).index(sku) + 1) % len(shuffled)]

    sub_price = panel[["sku_id", "date", "price"]].rename(
        columns={"sku_id": "substitute_id", "price": "substitute_price"}
    )
    panel = panel.copy()
    panel["substitute_id"] = panel["sku_id"].map(substitute_map)
    panel = panel.merge(sub_price, on=["substitute_id", "date"], how="left")
    panel["substitute_price_ratio"] = (panel["price"] / panel["substitute_price"]).round(3)
    # cannibalization_index: how much cheaper the substitute is, floored at 0
    panel["cannibalization_index"] = (1 - panel["substitute_price_ratio"]).clip(lower=0)
    return panel.drop(columns=["substitute_id", "substitute_price"])

### 6. Demand DGP -- combine every regressor into true_demand

In [23]:
def simulate_true_demand(panel: pd.DataFrame, sku_master: pd.DataFrame, calendar: pd.DataFrame,
                          weather: pd.DataFrame) -> pd.DataFrame:
    df = panel.merge(sku_master, on="sku_id", suffixes=("", "_master"))
    df = df.merge(calendar, on="date").merge(weather, on="date")
    df = df.sort_values(["sku_id", "date"]).reset_index(drop=True)

    profile = df["category"].map(CATEGORY_PROFILE)
    df["pattern_type"] = profile.apply(lambda p: p["pattern_type"])
    amplitude = profile.apply(lambda p: p["amplitude"])
    peak_week = profile.apply(lambda p: p["peak_week"] if p["peak_week"] is not None else 26)
    weather_sign = profile.apply(lambda p: p["weather_sign"])
    elasticity = df["category"].map(PRICE_ELASTICITY)

    # week index since series start, used for trend and long cyclic components
    week_idx = (df["date"] - df["date"].min()).dt.days // 7

    # -- trend component: only "trend" and "trend_seasonal" categories drift
    is_trend_like = df["pattern_type"].isin(["trend", "trend_seasonal"])
    trend = 1 + is_trend_like * 0.0015 * week_idx  # slow ~0.15%/week compounding growth

    # -- seasonal component: annual sinusoid keyed to each category's peak week
    seasonal = 1 + amplitude * np.sin(2 * np.pi * (df["week_of_year"] - peak_week + 13) / 52)
    seasonal = np.where(df["pattern_type"].isin(["seasonal", "trend_seasonal"]), seasonal, 1.0)

    # -- cyclic component: ~2-year business-cycle-like wave, unrelated to the
    # calendar -- this is what should make "cyclic" category models fail if
    # you only give them calendar/seasonal features, by design.
    cyclic = 1 + amplitude * np.sin(2 * np.pi * week_idx / 104)
    cyclic = np.where(df["pattern_type"] == "cyclic", cyclic, 1.0)

    # -- price elasticity effect
    price_effect = (df["price"] / df["base_price"]) ** (-elasticity)

    # -- promo uplift, on top of the price effect itself
    promo_effect = np.where(df["promo_flag"], rng.uniform(1.3, 2.0, len(df)), 1.0)

    # -- weather effect: deviation from that week's seasonal norm, signed per
    # category (outerwear up when colder than normal, tops up when warmer)
    seasonal_norm_temp = 12 + 15 * np.sin(2 * np.pi * (df["week_of_year"] - 30) / 52)
    temp_deviation = (df["temperature"] - seasonal_norm_temp) / 10
    weather_effect = 1 + weather_sign * 0.15 * temp_deviation

    # -- marketing: diminishing-returns (log) uplift
    marketing_effect = 1 + 0.08 * np.log1p(df["marketing_spend"] / 500)

    # -- cannibalization: cheaper substitute pulls demand down
    cannibal_effect = 1 - 0.3 * df["cannibalization_index"]

    # -- lifecycle: ramp up over first 8 weeks post-launch, slow decline
    # after week 150 of a SKU's own life (mirrors real product lifecycles)
    weeks_since_launch = week_idx - df["launch_week"]
    ramp = np.clip(weeks_since_launch / 8, 0, 1)
    decline = np.where(weeks_since_launch > 150, 1 - 0.002 * (weeks_since_launch - 150), 1.0)
    lifecycle_effect = np.clip(ramp * decline, 0, None)

    noise = rng.lognormal(mean=0, sigma=0.25, size=len(df))

    true_demand = (
        df["base_demand_level"] * trend * seasonal * cyclic * price_effect
        * promo_effect * weather_effect * marketing_effect * cannibal_effect
        * lifecycle_effect * noise
    )
    # rows before a SKU's own launch shouldn't exist at all -- drop them
    # instead of zeroing, since a true zero is meaningfully different from
    # "not on sale yet".
    df["true_demand"] = np.round(true_demand).clip(lower=0)
    df["days_since_launch"] = weeks_since_launch * 7
    df = df[weeks_since_launch >= 0].reset_index(drop=True)
    return df

### 7. Inventory simulation -> stockout censoring -> sales_qty

In [28]:
def simulate_inventory(df: pd.DataFrame) -> pd.DataFrame:

    df = df.sort_values(["sku_id", "date"]).copy()
    sales_qty = np.zeros(len(df))
    inventory_on_hand = np.zeros(len(df))
    stockout_flag = np.zeros(len(df), dtype=bool)

    for sku, idx in df.groupby("sku_id").indices.items():
        idx = np.array(idx)
        demand = df["true_demand"].values[idx]
        lead_weeks = max(1, int(df["lead_time_days"].values[idx[0]] / 7))

        target_cover_weeks = 4
        stock = demand[:lead_weeks].mean() * target_cover_weeks if len(demand) else 0
        for t in range(len(idx)):
            if t > 0 and t % lead_weeks == 0:
                trailing = demand[max(0, t - 8):t].mean() if t > 0 else demand[0]
                order_qty = trailing * target_cover_weeks * rng.uniform(0.75, 1.05)
                stock += order_qty

            sold = min(demand[t], stock)
            sales_qty[idx[t]] = sold
            inventory_on_hand[idx[t]] = stock
            stockout_flag[idx[t]] = demand[t] > stock
            stock = max(0, stock - sold)

    df["sales_qty"] = np.round(sales_qty)
    df["inventory_on_hand"] = np.round(inventory_on_hand)
    df["stockout_flag"] = stockout_flag
    return df

### 8. Endogenous feedback (reviews, ratings, wishlist, trend score)

In [25]:
def simulate_endogenous_feedback(df: pd.DataFrame) -> pd.DataFrame:

    df = df.sort_values(["sku_id", "date"]).copy()
    df["review_count"] = np.round(df["sales_qty"] * rng.uniform(0.05, 0.15, len(df))).clip(0)
    df["avg_rating"] = (df["quality_score"] + rng.normal(0, 0.15, len(df))).clip(1, 5).round(1)
    df["wishlist_adds"] = np.round(df["sales_qty"] * rng.uniform(0.2, 0.6, len(df))
                                    + rng.poisson(3, len(df))).clip(0)

    trend_base = df.groupby("sku_id")["sales_qty"].transform(
        lambda s: s.rolling(4, min_periods=1).mean()
    )
    df["social_trend_score"] = (trend_base / (trend_base.max() + 1e-6) * 100
                                 + rng.normal(0, 5, len(df))).clip(0, 100).round(1)
    return df

## 9. Assemble

In [26]:
def build_dataset():
    sku_master = make_sku_master(N_SKUS)
    calendar = make_calendar(START_DATE, N_WEEKS)
    weather = make_weather(calendar)

    panel = make_price_promo_marketing(sku_master, calendar)
    panel = add_cross_product_features(panel, sku_master)

    df = simulate_true_demand(panel, sku_master, calendar, weather)
    df = simulate_inventory(df)
    df = simulate_endogenous_feedback(df)

    full_cols = [
        "sku_id", "date", "category", "sub_category", "color", "material", "brand",
        "gender", "size_range", "price", "discount_pct", "promo_flag", "markdown_stage",
        "competitor_price_index", "week_of_year", "month", "quarter", "is_holiday",
        "is_weekend", "fashion_week_flag", "back_to_school_flag", "payday_week_flag",
        "days_since_launch", "temperature", "precipitation", "marketing_spend",
        "campaign_flag", "lead_time_days", "store_count", "channel",
        "review_count", "avg_rating", "wishlist_adds", "social_trend_score",
        "inventory_on_hand", "stockout_flag", "cannibalization_index",
        "substitute_price_ratio", "pattern_type", "true_demand", "sales_qty",
    ]
    full = df[full_cols].reset_index(drop=True)

    model_ready = full.drop(columns=["pattern_type", "true_demand"])
    return full, model_ready

Dataset creation step

In [29]:
full, model_ready = build_dataset()

In [30]:
print(f"full shape: {full.shape}")

full shape: (70311, 41)


In [31]:
print(f"model_ready shape: {model_ready.shape}")

model_ready shape: (70311, 39)


In [32]:
print(f"SKUs: {full['sku_id'].nunique()}, date range: {full['date'].min()} to {full['date'].max()}")
print(f"overall stockout rate: {full['stockout_flag'].mean():.2%}")
print(f"mean (true_demand - sales_qty) on stockout rows: "
      f"{(full.loc[full.stockout_flag, 'true_demand'] - full.loc[full.stockout_flag, 'sales_qty']).mean():.2f}")

SKUs: 300, date range: 2020-01-06 00:00:00 to 2024-12-23 00:00:00
overall stockout rate: 15.48%
mean (true_demand - sales_qty) on stockout rows: 28.18


# BigQuery - Table Data Source

### setup

In [33]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
print(PROJECT_ID)

REGION = 'us-central1'

# source data
BQ_PROJECT = PROJECT_ID
BQ_DATASET = 'input'
BQ_TABLE = 'synthetic_fashion_demand'

# Data source
BQ_SOURCE = model_ready

# clients
bq = bigquery.Client(project = PROJECT_ID)
gcs = storage.Client(project = PROJECT_ID)

# params
BUCKET = PROJECT_ID

fashionmvforecast


In [34]:
file = f"data/input/{BQ_TABLE}.csv"
bucketDef = gcs.bucket(BUCKET)
gcs_uri = f"gs://{BUCKET}/{file}"

In [35]:
if storage.Blob(bucket = bucketDef, name = file).exists(gcs):
    print(f'The file has already been created at: gs://{bucketDef.name}/{file}')
else:
    model_ready.to_csv(gcs_uri, index=False)
    print(f'Exported the table to: gs://{bucketDef.name}/{file}')

The file has already been created at: gs://fashionmvforecast/data/input/synthetic_fashion_demand.csv


# Generate a parameter YAML file for all future codes

In [36]:
import yaml
from google.cloud import storage

# 1. Define your bucket and target file path (blob name)
BUCKET_NAME = BUCKET
BLOB_PATH = "config/params.yaml"  # Path inside the bucket

# 2. Your YAML configuration as a Python dictionary
params_data = {
    "data_generation": {
        "seed": 42,
        "n_skus": 300,
        "start_date": "2020-01-06",
        "n_weeks": 260
    },
    "features": {
        "granularity": "W",
        "forecast_horizon": 8,
        "validation_horizon": 104
    },
    "baseline_model": {
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_data_in_leaf": 50,
        "num_boost_round": 1000,
        "early_stopping_rounds": 50
    },
    "tuning": {
        "seed": 42,
        "n_trials_default": 5
    }
}

# 3. Convert dictionary to YAML string format
yaml_content = yaml.dump(params_data, sort_keys=False)

# 4. Upload to GCS
client = storage.Client(project=BUCKET_NAME)
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(BLOB_PATH)

# Upload the string directly as a text file
blob.upload_from_string(yaml_content, content_type="text/x-yaml")

print(f"Successfully uploaded params.yaml to: gs://{BUCKET_NAME}/{BLOB_PATH}")

Successfully uploaded params.yaml to: gs://fashionmvforecast/config/params.yaml
